# 10. cal01_base_v5 만들기

| 항목 | 내용 |
| --- | --- |
| 입력 | `cal01_base_v3.csv` (원기록) |
| 출력 | `data/processed/cal01_base_v5.csv` (1분 묶음) |
| 이 버전에서 하는 일 | P제철 가이드대로 LOT 안에서 1분씩 묶음 |

**1분으로 묶는 규칙**

| 컬럼 | 1분 값 |
| --- | --- |
| 센서 (예: `TC-Z4`) | 1분 평균 |
| `센서__MIN`, `센서__MAX`, `센서__STD` | 1분 안의 최소, 최대, 표준편차 |
| `ROW_COUNT` | 1분 안의 원기록 행 수 |
| `EXCLUDED` | 1분 안에 제외 행이 하나라도 있으면 True |

**라벨 규칙**

| 컬럼 | 1 | 0 | 빈칸 |
| --- | --- | --- | --- |
| `구역__STAT_LABEL` | 1분 안에 원기록 STAT 1이 하나라도 있음 | 모두 0 | 제외한 분, 판단할 행이 없음 |
| `구역__DOMAIN_LABEL` | 1분 안에 DOMAIN 1이 하나라도 있음 | STAT 1인 원기록이 모두 해제됨 | 그 밖 |

**v1 ~ v5 한눈에 보기** (버전마다 한 가지씩만 더함)

| 버전 | 단위 | STAT_LABEL | DOMAIN_LABEL | 만드는 노트북 |
| --- | --- | --- | --- | --- |
| v1 | 원기록 (1초) | 같은 LOT 기준 \|Z\| > 3 | 없음 | 06 |
| v2 | 원기록 (1초) | v1과 같음 | 공통 움직임 검토로 해제한 곳 0 | 07 |
| v3 | 원기록 (1초) | v1과 같음 | v2 + LOT 종료부 검토로 해제한 곳 0 | 08 |
| v4 | 1분 | 1분 안에 후보가 하나라도 있으면 1 | v2를 1분으로 묶음 | 09 |
| v5 | 1분 | v4와 같음 | v3를 1분으로 묶음 | 10 |

## 1. 공통 경로

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v5)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v3.csv"  # v3를 읽음
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 만들 CSV
OUTPUT_FILE = "cal01_base_v5.csv"
OUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)

import numpy as np

## 2. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

## 3. 원기록 읽기

In [ ]:
raw = pd.read_csv(BASE_CSV, encoding=ENC, low_memory=False)  # low_memory=False: 컬럼 형식을 파일 전체를 보고 정함
raw["MEAS_DT"] = pd.to_datetime(raw["MEAS_DT"], errors="raise")

# 초를 버리고 분까지만 남김 (예: 10:50:37 -> 10:50:00)
raw["MINUTE"] = raw["MEAS_DT"].dt.floor("min")

print("원기록:", raw.shape)

## 4. 원기록 행마다 도메인 표시 만들기

| 표시 | True인 경우 |
| --- | --- |
| `구역_D1` | DOMAIN_LABEL이 1 |
| `구역_OPEN` | STAT_LABEL이 1인데 DOMAIN_LABEL이 빈칸 (아직 해제되지 않은 후보) |

In [ ]:
for zone in GROUPS:
    stat = raw[zone + "__STAT_LABEL"]
    domain = raw[zone + "__DOMAIN_LABEL"]

    raw[zone + "_D1"] = domain == 1
    raw[zone + "_OPEN"] = (stat == 1) & domain.isna()

## 5. 1분씩 묶기

In [ ]:
grouped = raw.groupby(["LOT_NO", "MINUTE"])

base = grouped["GRADE"].first().reset_index()
base = base.rename(columns={"MINUTE": "MEAS_DT"})

base["ROW_COUNT"] = grouped["MEAS_DT"].count().values
base["EXCLUDED"] = grouped["EXCLUDED"].max().values
base["EXCLUSION_REASON"] = grouped["EXCLUSION_REASON"].first().values

for sensor in SAVE_SENSORS:
    base[sensor] = grouped[sensor].mean().values
    base[sensor + "__MIN"] = grouped[sensor].min().values
    base[sensor + "__MAX"] = grouped[sensor].max().values
    base[sensor + "__STD"] = grouped[sensor].std().values

# 컬럼을 하나씩 많이 붙였으므로 표를 한 번 정리함 (속도 경고 방지)
base = base.copy()

print("1분 묶음:", base.shape)

## 6. 1분 라벨 붙이기

In [ ]:
for zone in GROUPS:
    stat_col = zone + "__STAT_LABEL"
    domain_col = zone + "__DOMAIN_LABEL"

    has_one = grouped[zone + "_D1"].max().values
    has_open = grouped[zone + "_OPEN"].max().values

    # 1분 안에 STAT 1이 하나라도 있으면 1
    base[stat_col] = grouped[stat_col].max().values

    base[domain_col] = np.nan
    base.loc[(base[stat_col] == 1) & (has_open == False), domain_col] = 0
    base.loc[has_one == True, domain_col] = 1

# 제외한 분은 라벨을 비워둠
for column in LABEL_COLUMNS:
    base.loc[base["EXCLUDED"] == True, column] = np.nan

base = base.copy()

## 7. 저장

| 컬럼 묶음 | 컬럼 |
| --- | --- |
| 식별 | `ROW_ID`, `LOT_NO`, `MEAS_DT` (1분 시작 시각), `GRADE`, `ROW_COUNT` |
| 제외 | `EXCLUDED`, `EXCLUSION_REASON` |
| 센서 22개 x 4 | 센서(평균), `__MIN`, `__MAX`, `__STD` |
| 라벨 22개 | `구역__STAT_LABEL`, `구역__DOMAIN_LABEL` |

In [ ]:
# 행 이름: LOT_분 (예: 240001_20240102T0336)
base["ROW_ID"] = base["LOT_NO"].astype(str) + "_" + base["MEAS_DT"].dt.strftime("%Y%m%dT%H%M")

# 라벨을 1.0 대신 1로 저장하려고 정수형(Int64)으로 바꿈. Int64는 빈칸도 담을 수 있음
for column in LABEL_COLUMNS:
    base[column] = base[column].astype("Int64")

sensor_columns = []
for sensor in SAVE_SENSORS:
    sensor_columns = sensor_columns + [sensor, sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]

columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "ROW_COUNT", "EXCLUDED", "EXCLUSION_REASON"] + sensor_columns + LABEL_COLUMNS
base = base[columns]

base.to_csv(OUT_CSV, index=False, encoding=ENC)
print("저장:", OUTPUT_FILE, base.shape)

## 8. 구역별 라벨 개수

In [ ]:
# 구역별 개수
rows = []

for zone in GROUPS:
    stat = base[zone + "__STAT_LABEL"]
    domain = base[zone + "__DOMAIN_LABEL"]

    rows.append({
        "구역": zone,
        "STAT 1": int((stat == 1).sum()),
        "STAT 0": int((stat == 0).sum()),
        "STAT 빈칸": int(stat.isna().sum()),
        "DOMAIN 1": int((domain == 1).sum()),
        "DOMAIN 0": int((domain == 0).sum()),
    })

zone_counts = pd.DataFrame(rows)
zone_counts

## 9. LOT별 통계 후보 개수 (분 단위)

In [ ]:
# LOT별로 구역마다 STAT_LABEL이 1인 개수
stat_columns = []
for zone in GROUPS:
    stat_columns.append(zone + "__STAT_LABEL")

lot_counts = base.groupby("LOT_NO")[stat_columns].sum()
lot_counts.columns = list(GROUPS)
lot_counts